# EgoLife CLIP Anchor and Evidence-Gap Demo

This notebook runs a small preprocessing experiment on one synchronized pair of EgoLife videos.

The experiment does **not** generate questions. It first tests whether CLIP retrieval can discover useful multi-user evidence:

1. Download one aligned two-user evidence packet using the current repository pipeline.
2. Sample a short window from each video.
3. Encode frames with CLIP.
4. Group redundant frames within each user using cosine K-means.
5. Find mutual-nearest shared anchors across users.
6. Rank user-specific evidence gaps by cross-user novelty.

> A high novelty score means no close visual match was found in the other user's sampled representatives. It does not prove that the other user never saw the event or object.

## 1. Select a GPU Runtime

In Colab, choose **Runtime → Change runtime type → T4 GPU**. CLIP can run on CPU, but a T4 makes model loading and embedding faster.

## 2. Upload the Project Directory and Install Dependencies

Before running this cell, compress your local `egolife_two_user_qa` directory into `egolife_two_user_qa.zip`. The ZIP should contain the package directory itself, including `__init__.py`, `clip_gap_demo.py`, `cli.py`, and `notebooks/`.

This setup uploads and extracts that directory directly into Colab. No GitHub repository or branch is required.

In [ ]:
import os
import sys
import shutil
import zipfile
from pathlib import Path
from google.colab import files

UPLOAD_ROOT = Path("/content/uploaded_egolife_project")
PACKAGE_DIR = UPLOAD_ROOT / "egolife_two_user_qa"

# Make the setup cell safe to rerun after a previous execution changed directories.
os.chdir("/content")
if UPLOAD_ROOT.exists():
    shutil.rmtree(UPLOAD_ROOT)
UPLOAD_ROOT.mkdir(parents=True, exist_ok=True)

print("Upload egolife_two_user_qa.zip")
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
assert zip_names, "Please upload a ZIP containing the egolife_two_user_qa directory."

zip_path = Path(zip_names[0])
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(UPLOAD_ROOT)

# Accept either ZIP layout:
#   egolife_two_user_qa/__init__.py
# or files placed directly at the ZIP root.
if not PACKAGE_DIR.exists() and (UPLOAD_ROOT / "__init__.py").exists():
    direct_files = [path for path in UPLOAD_ROOT.iterdir() if path.name != "egolife_two_user_qa"]
    PACKAGE_DIR.mkdir()
    for path in direct_files:
        shutil.move(str(path), PACKAGE_DIR / path.name)

assert (PACKAGE_DIR / "__init__.py").exists(), (
    "Could not find egolife_two_user_qa/__init__.py in the uploaded ZIP."
)

%cd {UPLOAD_ROOT}
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!pip install -q -U "transformers>=4.57,<5" huggingface_hub "pillow>=10,<12" matplotlib

sys.path.insert(0, str(UPLOAD_ROOT))
print("Uploaded package:", PACKAGE_DIR)
print("Python package parent:", UPLOAD_ROOT)


The following check catches two common setup problems: an incompatible Torch/Torchvision installation, or uploading an older directory that does not contain `clip_gap_demo.py`.

In [ ]:
import shutil
import torch
import torchvision
import transformers
from transformers import CLIPModel, CLIPProcessor

clip_demo_path = PACKAGE_DIR / "clip_gap_demo.py"
assert clip_demo_path.exists(), (
    f"{clip_demo_path} is missing. Recreate the ZIP from the updated local egolife_two_user_qa directory."
)

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("FFmpeg:", shutil.which("ffmpeg"))
print("CLIP demo:", clip_demo_path)


## 3. Configure the Toy Experiment

The defaults search Days 1–5 and all six EgoLife users. They select five synchronized instances, stratified so that the initial experiment contains one random instance per day. If more than two users share an instance, the evidence builder also samples a random user pair.

In [ ]:
from pathlib import Path

DAYS = "DAY1,DAY2,DAY3,DAY4,DAY5"
AGENTS = "A1_JAKE,A2_ALICE,A3_TASHA,A4_LUCIA,A5_KATRINA,A6_SHURE"
MAX_PER_AGENT_DAY = 30
NUM_DIVERSE_INSTANCES = 5
RANDOM_SEED = 42

START_SECONDS = 0.0
DURATION_SECONDS = 12.0
SAMPLE_INTERVAL_SECONDS = 1.5
CLUSTERS_PER_USER = 4
ANCHOR_THRESHOLD = 0.75
TOP_K = 3
MODEL_ID = "openai/clip-vit-base-patch32"

OUTPUT_ROOT = PACKAGE_DIR / "outputs" / "clip_gap_colab"
CACHE_DIR = OUTPUT_ROOT / "hf_cache"
MANIFEST = OUTPUT_ROOT / "manifest.json"
EVIDENCE = OUTPUT_ROOT / "evidence.jsonl"
MINED_DIR = OUTPUT_ROOT / "mined"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Output root:", OUTPUT_ROOT.resolve())
print("Days:", DAYS)
print("Users:", AGENTS)
print("Diverse instances:", NUM_DIVERSE_INSTANCES)
print("Window duration per instance:", DURATION_SECONDS, "seconds")


## 4. Optional Hugging Face Login

The datasets and CLIP model are normally public. Run this only if Hugging Face requests authentication or rate-limits the download.

In [ ]:
LOGIN_TO_HUGGING_FACE = False

if LOGIN_TO_HUGGING_FACE:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print("Skipping Hugging Face login.")


## 5. Build a Small Aligned Manifest

This uses the current repository's manifest logic and searches for matching video and gaze files.

In [ ]:
!python -m egolife_two_user_qa build_manifest \
  --days {DAYS} \
  --agents {AGENTS} \
  --max-per-agent-day {MAX_PER_AGENT_DAY} \
  --no-overlays \
  --output {MANIFEST}


In [ ]:
import json

manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
print(json.dumps(manifest.get("summary", {}), indent=2))
print("First clips:")
print(json.dumps(manifest.get("clips", [])[:2], indent=2)[:3000])


## 6. Prepare Five Diverse Two-User Evidence Packets

This randomly selects five synchronized groups across Days 1–5, with one group per day for the first five packets. It downloads the selected videos and gaze files. The same random seed also randomizes the selected user pair when more than two users are available.

In [ ]:
!python -m egolife_two_user_qa prepare_evidence \
  --manifest {MANIFEST} \
  --output {EVIDENCE} \
  --output-root {OUTPUT_ROOT} \
  --cache-dir {CACHE_DIR} \
  --target-count {NUM_DIVERSE_INSTANCES} \
  --users-per-case 2 \
  --frames-per-clip 4 \
  --random-seed {RANDOM_SEED} \
  --stratify-by-day


In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

packets = read_jsonl(EVIDENCE)
assert len(packets) >= NUM_DIVERSE_INSTANCES, (
    f"Only {len(packets)} packets were found. Increase MAX_PER_AGENT_DAY or inspect other days."
)
packet = packets[0]  # retained only for the optional single-packet baseline below
for index, row in enumerate(packets, 1):
    print(
        f"{index}. {row.get('day')} {row.get('clip_clock')}",
        "users=", row.get("required_users"),
        "evidence_id=", row.get("evidence_id"),
    )


## 7. Optional Single-Window Baseline

The original experiment used only the first 12 seconds. It is now disabled by default so running the entire notebook performs exactly five randomized trials in Section 11. Set `RUN_SINGLE_WINDOW_BASELINE = True` only if you also want the original baseline.

In [ ]:
from egolife_two_user_qa.clip_gap_demo import run_clip_gap_demo

RUN_SINGLE_WINDOW_BASELINE = False
baseline_results = None

if RUN_SINGLE_WINDOW_BASELINE:
    baseline_results = run_clip_gap_demo(
        evidence_path=EVIDENCE,
        output_dir=MINED_DIR,
        model_id=MODEL_ID,
        start_seconds=START_SECONDS,
        duration_seconds=DURATION_SECONDS,
        sample_interval_seconds=SAMPLE_INTERVAL_SECONDS,
        clusters_per_user=CLUSTERS_PER_USER,
        anchor_threshold=ANCHOR_THRESHOLD,
        top_k=TOP_K,
    )
    print("Single-window baseline complete.")
else:
    print("Skipping the optional baseline. Continue to Section 11 for five randomized trials.")


## 8. Inspect the Optional Baseline Contact Sheet

In [ ]:
from IPython.display import Image as DisplayImage, display

CONTACT_SHEET = MINED_DIR / "clip_gap_contact_sheet.jpg"
if RUN_SINGLE_WINDOW_BASELINE and CONTACT_SHEET.exists():
    display(DisplayImage(filename=str(CONTACT_SHEET)))
else:
    print("Optional baseline was not run.")


## 9. Inspect the Optional Baseline Results

Shared anchors require mutual nearest-neighbor matching and the configured similarity threshold. Evidence gaps are ranked by `novelty = 1 - closest_cross_user_similarity`.

In [ ]:
RESULTS_PATH = MINED_DIR / "clip_gap_results.json"
if RUN_SINGLE_WINDOW_BASELINE and RESULTS_PATH.exists():
    results = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
    print("Users:", results["left_user"], "vs", results["right_user"])
    print("Model:", results["model_id"])
    print("Anchors:", len(results.get("anchors", [])))
    print(results["interpretation_warning"])
else:
    results = None
    print("Optional baseline was not run.")


## 10. Visualize the Optional Baseline Similarity Matrix

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if results is not None:
    matrix = np.asarray(results["similarity_matrix"])
    left_frames = results["representative_frames"][results["left_user"]]
    right_frames = results["representative_frames"][results["right_user"]]
    fig, ax = plt.subplots(figsize=(7, 5))
    image = ax.imshow(matrix, vmin=0, vmax=1, cmap="viridis")
    ax.set_xlabel(results["right_user"] + " representative timestamp")
    ax.set_ylabel(results["left_user"] + " representative timestamp")
    ax.set_xticks(range(len(right_frames)), [f"{f['timestamp_seconds']}s" for f in right_frames])
    ax.set_yticks(range(len(left_frames)), [f"{f['timestamp_seconds']}s" for f in left_frames])
    fig.colorbar(image, ax=ax, label="CLIP cosine similarity")
    plt.tight_layout()
    plt.show()
else:
    print("Optional baseline was not run.")


## 11. Run Five Diverse Instances Across Days 1–5

Each trial now uses a different evidence packet rather than another window from the same 30-second scene. The five packets were selected across Days 1–5 in Section 6. Within each packet, the experiment randomly selects one valid 12-second start time.

CLIP is loaded once and reused for all five instances. Each trial writes its own JSON result and contact sheet, while `diverse_packet_trials_summary.json` stores the aggregate experiment.

In [ ]:
from egolife_two_user_qa.clip_gap_demo import run_diverse_packet_trials
import pandas as pd

DIVERSE_TRIALS_DIR = OUTPUT_ROOT / "diverse_instance_trials"

aggregate = run_diverse_packet_trials(
    evidence_path=EVIDENCE,
    output_dir=DIVERSE_TRIALS_DIR,
    model_id=MODEL_ID,
    trial_count=NUM_DIVERSE_INSTANCES,
    random_seed=RANDOM_SEED,
    duration_seconds=DURATION_SECONDS,
    sample_interval_seconds=SAMPLE_INTERVAL_SECONDS,
    clusters_per_user=CLUSTERS_PER_USER,
    anchor_threshold=ANCHOR_THRESHOLD,
    top_k=TOP_K,
)

trial_results = aggregate["trials"]
trial_df = pd.DataFrame(aggregate["trial_summaries"])
trial_df.to_csv(DIVERSE_TRIALS_DIR / "diverse_instance_summary.csv", index=False)

print("Random seed:", RANDOM_SEED)
display(trial_df[[
    "trial", "day", "time_token", "users", "start_seconds", "end_seconds",
    "mean_cross_user_similarity", "max_anchor_similarity", "anchor_count",
    "largest_user_novelty", "review_priority",
]].round(3))


## 12. Plot the Pattern Across Diverse Instances

- Consistently high mean similarity and low novelty means both cameras remain in strongly related scenes.
- A novelty spike suggests one user's representatives become less visually matched by the other user.
- A strong anchor score in the same window indicates that shared context still exists.
- `review_priority` multiplies the largest user novelty by the best anchor similarity. It is only a heuristic for deciding which contact sheets to inspect first.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
trial_labels = [f"{row.day}\n{row.users}" for row in trial_df.itertuples()]
x = range(len(trial_df))
axes[0].plot(x, trial_df["mean_cross_user_similarity"], marker="o", label="mean cross-user similarity")
axes[0].plot(x, trial_df["max_anchor_similarity"], marker="s", label="best anchor similarity")
axes[0].set_ylabel("CLIP cosine similarity")
axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].plot(x, trial_df["largest_user_novelty"], marker="o", label="largest user novelty")
axes[1].axhline(0.20, color="red", linestyle="--", label="0.20 exploratory threshold")
axes[1].set_xlabel("different day / user instance")
axes[1].set_ylabel("novelty = 1 - best match")
axes[1].set_ylim(0, max(0.35, trial_df["largest_user_novelty"].max() + 0.05))
axes[1].grid(alpha=0.25)
axes[1].legend()
axes[1].set_xticks(list(x), trial_labels)
plt.tight_layout()
plt.show()


## 13. Inspect Every Trial Contact Sheet

The numerical score is only a retrieval signal. Inspect the images to determine whether a novelty spike reflects a meaningful object/action difference or merely camera angle, motion blur, lighting, or wearer position.

In [ ]:
for trial in trial_results:
    sheet = Path(trial["output_dir"]) / "clip_gap_contact_sheet.jpg"
    start = trial["window"]["start_seconds"]
    end = start + trial["window"]["duration_seconds"]
    print(f"Trial {trial['trial']}: {start:.2f}s to {end:.2f}s")
    display(DisplayImage(filename=str(sheet)))


## 14. Rank Windows for Manual Review

This deliberately simple triage score rewards maximum novelty while also requiring anchor strength. It only determines which contact sheets to inspect first.

In [ ]:
ranked = trial_df.sort_values("review_priority", ascending=False)
display(ranked[[
    "trial", "day", "users", "start_seconds", "end_seconds",
    "max_anchor_similarity", "anchor_count", "largest_user_novelty", "review_priority",
]].round(3))


## 15. Suggested Iteration

Try changing these parameters and rerun Sections 11–14:

- `RANDOM_SEED`: choose different synchronized instances, user pairs, and short-window starts.
- `DURATION_SECONDS`: 8–12 seconds is a useful toy range. Shorter windows overlap less.
- `SAMPLE_INTERVAL_SECONDS`: lower values add temporal detail but increase redundancy.
- `CLUSTERS_PER_USER`: increase if distinct moments are being merged.
- `ANCHOR_THRESHOLD`: lower it if no anchors appear; raise it if anchors look unrelated.

The next project step is to give the retrieved frames to a lightly prompted VLM and ask it to describe relationships before generating any MCQ.

## 16. Download the Results

In [ ]:
import shutil

archive = shutil.make_archive("clip_gap_colab_results", "zip", root_dir=OUTPUT_ROOT)
print("Archive:", archive)

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Not running in Colab; download manually from:", archive)
